# Create DB

In [1]:
# from data.ingestion import FnBDataIngestor
# from data.embeddings import Voyage4NanoEmbedding
# from data.documents import QdrantDocumentStore, get_qdrant_client
# from data.scoring import ProductScorer

# qdrant_client = get_qdrant_client(path="external_data_storage/qdrant_db")
# collection_name="fnb_menu"

# embedding = Voyage4NanoEmbedding()
# store = QdrantDocumentStore(collection_name=collection_name, qdrant_client=qdrant_client)
# scorer = ProductScorer(threshold=60)

# # Init ingestor (inject all dependencies)
# ingestor = FnBDataIngestor(
#         document_store=store,
#         embedding=embedding,
#         scorer=scorer,
#     )

# ingestor.ingest_from_file(file_path="external_data_storage/fnb.json", collection_name=collection_name)

# Agent

In [8]:
import multiprocessing.popen_spawn_posix
from typing import Any
from endpoints.schemas import (
    ChatRequest,
    ChatResponse,
    StreamDone,
    StreamError,
    StreamNodeUpdate,
    StreamToken,
)
from utils.logger import setup_logger

logger = setup_logger(None)

def _get_agent():
    """Lazy-load agent singleton — tránh circular imports & startup delay."""
    from endpoints._agent_singleton import get_agent

    return get_agent()

In [3]:
agent = _get_agent()

/Users/hung/Desktop/scanx-agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-03-12 17:05:03,776 - endpoints._agent_singleton - INFO - Initializing Agent singleton...
2026-03-12 17:05:03,777 - services.llm_service - INFO - LLM Service initialized — temperature=0.1
2026-03-12 17:05:03,946 - data.embeddings.base_embedding - INFO - Embedding initialized — model=voyageai/voyage-4-nano, dims=2048
2026-03-12 17:05:03,962 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: mps
2026-03-12 17:05:03,962 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: voyageai/voyage-4-nano


Loading weights: 100%|██████████| 135/135 [00:00<00:00, 6395.34it/s]


2026-03-12 17:05:10,695 - sentence_transformers.SentenceTransformer - INFO - 2 prompts are loaded, with the keys: ['query', 'document']
2026-03-12 17:05:10,698 - data.embeddings.voyage_4_nano - INFO - Voyage4NanoEmbedding ready — model=voyageai/voyage-4-nano
2026-03-12 17:05:10,740 - data.documents.qdrant_store - INFO - QdrantDocumentStore initialized — collection=fnb_menu, vector_size=2048, distance=Dot
2026-03-12 17:05:10,741 - agents.fnb_agent - INFO - FnBAgent dependencies initialized — embedding=voyageai/voyage-4-nano, store=QdrantDocumentStore(collection='fnb_menu', vector_size=2048, distance=Dot), collection=fnb_menu
2026-03-12 17:05:10,757 - agents.base_agent - INFO - Agent initialized — tools=['menu_search', 'get_product_detail', 'get_recommendations'], model=profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x12f90b380> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x12f90be00> root_client=<openai.O

In [4]:
async def chat_stream(message: str, configurable: dict[str, Any]):
    """SSE streaming chat — trả về từng token real-time."""
    agent = _get_agent()

    logger.info(
        "Stream request — message='%s', configurable=%s",
        message[:100], configurable,
    )

    async def event_generator():
        """Generate SSE events từ agent astream."""
        try:
            async for event in agent.astream(
                message,
                configurable=configurable,
            ):
                if event["mode"] == "messages":
                    node_name = event.get("node", "unknown")

                    sse_event = StreamToken(
                        node=node_name,
                        content=event["content"],
                    )
                    yield sse_event

                elif event["mode"] == "updates":
                    node_name = event.get("node", "unknown")
                    if node_name == "model":
                        sse_event = StreamNodeUpdate(
                            node=node_name, 
                            state=event["state"]['model']
                        )
                    elif node_name == "tools":
                        sse_event = StreamNodeUpdate(
                            node=node_name, 
                            state=event["state"]["tools"]
                        )
                    else:
                        sse_event = StreamNodeUpdate(node=node_name, state=event)

                    yield sse_event

            # Stream hoàn tất
            done_event = StreamDone()
            yield done_event

        except Exception as e:
            logger.error("Stream error: %s", e)
            error_event = StreamError(detail=str(e))
            yield error_event

    return event_generator()

async def run(message: str, configurable: dict[str, Any]):
    generator = await chat_stream(message, configurable)
    async for event in generator:
        event_type = event.type
        if event_type == "token":
            print(event.content, end="", flush=True)
        elif event_type == "node_update":
            if event.node == "model":
                if event.state['messages'][-1].response_metadata['finish_reason'] == "stop":
                    print()
            print("---",event.state, "---")
        elif event_type == "done":
            print("---",event.message, "---")
        elif event_type == "error":
            print("---",event.detail, "---")
        else:
            print(event)

In [9]:
configurable = {
    "tenant_id": "tenant-1",
    "thread_id": "default",
}
await run("tôi muốn gì đó ngọt, không quá đặc?", configurable)
print("="*50)
await run("Nguyên liệu của món đầu tiên trên đó gồm những gì?", configurable)
print("="*50)
await run("tôi sẽ chọn món này", configurable)

2026-03-12 17:06:57,747 - root - INFO - Stream request — message='tôi muốn gì đó ngọt, không quá đặc?', configurable={'tenant_id': 'tenant-1', 'thread_id': 'default'}
2026-03-12 17:06:57,748 - agents.base_agent - INFO - Agent astream — message='tôi muốn gì đó ngọt, không quá đặc?', configurable={'tenant_id': 'tenant-1', 'thread_id': 'default'}
--- {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-oss:120b-cloud', 'system_fingerprint': 'fp_ollama', 'model_provider': 'openai'}, id='lc_run--019ce183-80df-7820-b8ff-8cd06f34b338', tool_calls=[{'name': 'menu_search', 'args': {'query': 'ngọt không quá đặc', 'top_k': 5}, 'id': 'call_rn7v0g75', 'type': 'tool_call'}], invalid_tool_calls=[])]} ---
2026-03-12 17:06:59,333 - tools.menu_search_tool - INFO - menu_search — query='ngọt không quá đặc', tenant_id=tenant-1, top_k=5


Batches:   0%|          | 0/1 [00:00<?, ?it/s]/Users/hung/.cache/huggingface/modules/transformers_modules/voyageai/voyage_hyphen_4_hyphen_nano/67fabc9bef010dabc5f6024aa1b1b6b93410426f/modeling_qwen3_bidirectional.py:62: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  "full_attention": create_causal_mask(
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.57it/s]

2026-03-12 17:06:59,559 - tools.menu_search_tool - INFO - menu_search — found 5 results


--- {'messages': [ToolMessage(content='--- Kết quả 1 (Score: 0.4213) ---\nID: item-024\nTên: Nước ép táo nguyên chất\nGiá: 55,000đ\nDanh mục: Đồ uống\nMô tả: Tên món: Nước ép táo nguyên chất\nDanh mục: Đồ uống\nMô tả: Vị ngọt tự nhiên của táo, không thêm đường. Rất tốt cho sức khỏe và phù hợp cho trẻ nhỏ.\nNguyên liệu: Táo tươi\nĐặc điểm: Lạnh, Không cồn, Tốt cho sức khỏe\nPhục vụ: Cả ngày\nGiá: 55,000đ\nTags: Lạnh, Không cồn, Tốt cho sức khỏe\nThời gian phục vụ: Cả ngày\n\n--- Kết quả 2 (Score: 0.3874) ---\nID: item-011\nTên: Salad hoa quả sốt sữa chua\nGiá: 95,000đ\nDanh mục: Khai vị\nMô tả: Tên món: Salad hoa quả sốt sữa chua\nDanh mục: Khai vị\nMô tả: Vị chua ngọt thanh mát, giúp giải ngấy cực hiệu quả khi ăn cùng quá nhiều đồ nướng béo.\nNguyên liệu: Táo, Lê, Dưa hấu, Sữa chua không đường\nĐặc điểm: Chay, Thanh mát, Không lactose\nPhục vụ: Cả ngày\nGiá: 95,000đ\nTags: Chay, Thanh mát, Không lactose\nThời gian phục vụ: Cả ngày\n\n--- Kết quả 3 (Score: 0.3227) ---\nID: item-023\nTên

/Users/hung/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/asyncio/selector_events.py:274: RuntimeWarning: coroutine 'run' was never awaited
  def _add_reader(self, fd, callback, *args):


 quá đặc” mà mình nghĩ bạn sẽ thích:

| Món | Giá | Đặc điểm nổi bật |
|---|---|---|
| **Nước ép táo nguyên chất** (ID: item-024) | 55 000đ | Vị ngọt tự nhiên của táo tươi, không thêm đường, mát lạnh – rất thích hợp cho mọi lứa tuổi. |
| **Salad hoa quả sốt sữa chua** (ID: item-011) | 95 000đ | Kết hợp táo, lê, dưa hấu cùng sữa chua không đường, vừa ngọt vừa chua thanh, nhẹ nhàng và giàu chất xơ. |
| **Trà đào sả đá** (ID: item-023) | 45 000đ | Trà đen pha với đào ngâm ngọt nhẹ, sả thơm mát – một đồ uống giải nhiệt vừa ngọt vừa không đặc. |

Bạn muốn mình cung cấp thêm chi tiết (nguyên liệu, cách phục vụ) cho món nào không, hay muốn đặt ngay một trong ba món trên? Nếu bạn chọn món chính, mình cũng có thể gợi ý một vài đồ uống hoặc món kèm phù hợp để tăng thêm hương vị cho bữa ăn. 🌿🍹
--- {'messages': [AIMessage(content='Chào bạn! 🌟 Dưới đây là một vài món “ngọt nhẹ, không quá đặc” mà mình nghĩ bạn sẽ thích:\n\n| Món | Giá | Đặc điểm nổi bật |\n|---|---|---|\n| **Nước ép táo nguyên chất*